In [ ]:
# =============================================================================
# ERASMUS MOBILITY DATA: ETER INSTITUTIONAL LINKAGE
# =============================================================================
# 
# This notebook links Erasmus mobility records to the European Tertiary 
# Education Register (ETER) to add institutional characteristics.
#
# MATCHING STRATEGY:
# - Location-based proximity matching (within 50km)
# - Fuzzy string matching on institution names (min 60% similarity)
# - Combined scoring (60% name similarity + 40% distance proximity)
#
# INPUT:  erasmus_with_nuts3_corrected.csv (output from notebook 02)
# OUTPUT: erasmus_with_eter.csv - mobility data with ETER institutional data
#
# REQUIREMENTS:
# - ETER database Excel file
# - fuzzywuzzy library for string matching
#
# ESTIMATED RUNTIME: 15-30 minutes (matching 45K+ unique institutions)
# =============================================================================

In [ ]:
# =============================================================================
# CELL 1: Import Libraries
# =============================================================================

import pandas as pd
import numpy as np
from math import radians, cos, sin, asin, sqrt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Install fuzzywuzzy if needed
try:
    from fuzzywuzzy import fuzz
except ImportError:
    print("Installing fuzzywuzzy...")
    import subprocess
    subprocess.run(['pip', 'install', 'fuzzywuzzy', 'python-Levenshtein'], check=True)
    from fuzzywuzzy import fuzz

print("✓ Libraries loaded successfully!")

In [ ]:
# =============================================================================
# CELL 2: Configuration - USER INPUT REQUIRED
# =============================================================================

# ==========================
# CONFIGURE THESE PATHS:
# ==========================

# Path to geocoded + corrected Erasmus data (output from notebook 02)
ERASMUS_CORRECTED_PATH = '/path/to/erasmus_with_nuts3_corrected.csv'

# Path to ETER database Excel file
# Download from: https://www.eter-project.com
ETER_DATABASE_PATH = '/path/to/eter-export.xlsx'

# Output directory
OUTPUT_DIR = '/path/to/output/directory/'

# ==========================
# MATCHING PARAMETERS:
# ==========================

# Maximum distance for potential matches (km)
MAX_DISTANCE_KM = 50

# Minimum name similarity for valid match (0-100)
MIN_SIMILARITY = 60

# Combined score weights
WEIGHT_NAME = 0.6  # 60% weight on name similarity
WEIGHT_DISTANCE = 0.4  # 40% weight on proximity

print("=" * 80)
print("CONFIGURATION")
print("=" * 80)
print(f"Input data:  {ERASMUS_CORRECTED_PATH}")
print(f"ETER database: {ETER_DATABASE_PATH}")
print(f"Output dir:  {OUTPUT_DIR}")
print(f"\nMatching parameters:")
print(f"  Max distance: {MAX_DISTANCE_KM} km")
print(f"  Min similarity: {MIN_SIMILARITY}%")
print(f"  Scoring: {WEIGHT_NAME*100:.0f}% name + {WEIGHT_DISTANCE*100:.0f}% distance")
print("=" * 80)

In [ ]:
# =============================================================================
# CELL 3: Load Erasmus Data
# =============================================================================

print("=" * 80)
print("LOADING ERASMUS MOBILITY DATA")
print("=" * 80)

df_erasmus = pd.read_csv(ERASMUS_CORRECTED_PATH)
print(f"✓ Dataset loaded: {len(df_erasmus):,} records")

# Quick summary
print(f"\n📊 DATASET SUMMARY:")
print(f"  Total records: {len(df_erasmus):,}")
print(f"  Unique sending institutions: {df_erasmus['Sending Organization'].nunique():,}")
print(f"  Unique receiving institutions: {df_erasmus['Receiving Organization'].nunique():,}")
print(f"  Countries: {df_erasmus['Sending Country'].nunique()}")

# NUTS3 coverage
both_nuts3 = df_erasmus[
    (df_erasmus['Sending_NUTS3'].notna()) & 
    (df_erasmus['Receiving_NUTS3'].notna())
].shape[0]
print(f"  NUTS3 coverage: {both_nuts3:,} ({both_nuts3/len(df_erasmus)*100:.2f}%)")

In [ ]:
# =============================================================================
# CELL 4: Load ETER Database
# =============================================================================

print("\n" + "=" * 80)
print("LOADING ETER DATABASE")
print("=" * 80)

df_eter = pd.read_excel(ETER_DATABASE_PATH, sheet_name='full')
print(f"✓ ETER database loaded: {len(df_eter):,} records")

# Convert coordinates to numeric
df_eter['GEO.COORDLAT'] = pd.to_numeric(df_eter['GEO.COORDLAT'], errors='coerce')
df_eter['GEO.COORDLON'] = pd.to_numeric(df_eter['GEO.COORDLON'], errors='coerce')

# Filter to institutions with coordinates
df_eter_geo = df_eter[
    df_eter['GEO.COORDLAT'].notna() & 
    df_eter['GEO.COORDLON'].notna()
].copy()

print(f"✓ ETER with coordinates: {len(df_eter_geo):,}")

# Show ETER structure
print(f"\n📋 ETER DATABASE STRUCTURE:")
print(f"  Countries: {df_eter_geo['BAS.COUNTRY'].nunique()}")
print(f"  Reference years: {sorted(df_eter_geo['BAS.REFYEAR'].unique())}")
print(f"  Key variables available:")
print(f"    - BAS.ETERID: Unique institution ID")
print(f"    - BAS.INSTNAME: Institution name (local)")
print(f"    - BAS.INSTNAMEENGL: Institution name (English)")
print(f"    - GEO.NUTS3: NUTS3 code")
print(f"    - PERS.ACAFTETOTAL: Academic staff (FTE)")
print(f"    - REV.CORETOTAL.EURO: Core revenue")
print(f"    - PATPUBL.PUBLFULLTOT: Publications")

In [ ]:
# =============================================================================
# CELL 5: Prepare Unique Institution Lists
# =============================================================================

print("\n" + "=" * 80)
print("PREPARING UNIQUE INSTITUTION LISTS")
print("=" * 80)

# Extract unique sending institutions
sending_unique = df_erasmus[[
    'Sending Organization', 'Sending City', 'Sending Country', 
    'Sending_Lat', 'Sending_Lon', 'Sending_NUTS3'
]].drop_duplicates()

sending_unique = sending_unique.rename(columns={
    'Sending Organization': 'Institution',
    'Sending City': 'City',
    'Sending Country': 'Country',
    'Sending_Lat': 'Latitude',
    'Sending_Lon': 'Longitude',
    'Sending_NUTS3': 'NUTS3'
})

# Extract unique receiving institutions
receiving_unique = df_erasmus[[
    'Receiving Organization', 'Receiving City', 'Receiving Country',
    'Receiving_Lat', 'Receiving_Lon', 'Receiving_NUTS3'
]].drop_duplicates()

receiving_unique = receiving_unique.rename(columns={
    'Receiving Organization': 'Institution',
    'Receiving City': 'City',
    'Receiving Country': 'Country',
    'Receiving_Lat': 'Latitude',
    'Receiving_Lon': 'Longitude',
    'Receiving_NUTS3': 'NUTS3'
})

# Combine and deduplicate
erasmus_institutions = pd.concat([sending_unique, receiving_unique]).drop_duplicates(
    subset=['Institution', 'City', 'Country']
).reset_index(drop=True)

# Add country code
erasmus_institutions['Country_Code'] = erasmus_institutions['Country'].str.split(' - ').str[0]

# Remove institutions without coordinates
erasmus_institutions = erasmus_institutions[
    erasmus_institutions['Latitude'].notna() & 
    erasmus_institutions['Longitude'].notna()
].copy()

print(f"✓ Unique Erasmus institutions: {len(erasmus_institutions):,}")
print(f"  With coordinates: {len(erasmus_institutions):,}")
print(f"  Countries represented: {erasmus_institutions['Country_Code'].nunique()}")

In [ ]:
# =============================================================================
# CELL 6: Define Matching Functions
# =============================================================================

def haversine(lon1, lat1, lon2, lat2):
    """
    Calculate great circle distance between two points in kilometers.
    
    Parameters:
    -----------
    lon1, lat1 : float
        Coordinates of first point
    lon2, lat2 : float
        Coordinates of second point
    
    Returns:
    --------
    float : Distance in kilometers
    """
    lon1, lat1, lon2, lat2 = map(radians, [float(lon1), float(lat1), 
                                            float(lon2), float(lat2)])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return 6371 * c  # Earth radius in km


def find_best_eter_match(erasmus_inst, df_eter_geo, 
                          max_distance_km=MAX_DISTANCE_KM, 
                          min_similarity=MIN_SIMILARITY):
    """
    Find best ETER match for an Erasmus institution.
    
    Matching strategy:
    1. Filter ETER to same country
    2. Calculate distances to all ETER institutions
    3. Keep only institutions within max_distance_km
    4. Calculate name similarity (fuzzy matching)
    5. Combine distance and similarity scores
    6. Return best match if above minimum similarity threshold
    
    Parameters:
    -----------
    erasmus_inst : pd.Series
        Erasmus institution record
    df_eter_geo : pd.DataFrame
        ETER database with coordinates
    max_distance_km : float
        Maximum distance for potential matches
    min_similarity : int
        Minimum name similarity (0-100)
    
    Returns:
    --------
    dict : Match results including ETER ID, similarity, distance
    """
    country_code = erasmus_inst['Country_Code']
    
    # Filter to same country
    eter_country = df_eter_geo[df_eter_geo['BAS.COUNTRY'] == country_code].copy()
    
    if len(eter_country) == 0:
        return {
            'eter_id': None,
            'eter_name_english': None,
            'eter_city': None,
            'eter_nuts3': None,
            'distance_km': None,
            'name_similarity': None,
            'combined_score': None,
            'match_method': 'no_eter_in_country'
        }
    
    # Calculate distances
    eter_country['distance_km'] = eter_country.apply(
        lambda row: haversine(
            erasmus_inst['Longitude'], erasmus_inst['Latitude'],
            row['GEO.COORDLON'], row['GEO.COORDLAT']
        ), axis=1
    )
    
    # Filter by distance
    eter_nearby = eter_country[eter_country['distance_km'] <= max_distance_km].copy()
    
    if len(eter_nearby) == 0:
        return {
            'eter_id': None,
            'eter_name_english': None,
            'eter_city': None,
            'eter_nuts3': None,
            'distance_km': None,
            'name_similarity': None,
            'combined_score': None,
            'match_method': 'no_nearby_eter'
        }
    
    # Calculate name similarity
    erasmus_name = str(erasmus_inst['Institution']).upper()
    
    def calc_similarity(row):
        local = str(row['BAS.INSTNAME']).upper() if pd.notna(row['BAS.INSTNAME']) else ""
        english = str(row['BAS.INSTNAMEENGL']).upper() if pd.notna(row['BAS.INSTNAMEENGL']) else ""
        return max(
            fuzz.token_sort_ratio(erasmus_name, local),
            fuzz.token_sort_ratio(erasmus_name, english)
        )
    
    eter_nearby['name_similarity'] = eter_nearby.apply(calc_similarity, axis=1)
    
    # Combined scoring (60% name, 40% distance)
    max_dist = eter_nearby['distance_km'].max()
    eter_nearby['distance_score'] = (
        100 * (1 - eter_nearby['distance_km'] / max_dist) if max_dist > 0 else 100
    )
    eter_nearby['combined_score'] = (
        WEIGHT_NAME * eter_nearby['name_similarity'] + 
        WEIGHT_DISTANCE * eter_nearby['distance_score']
    )
    
    # Get best match
    best = eter_nearby.nlargest(1, 'combined_score').iloc[0]
    
    # Reject if similarity too low
    if best['name_similarity'] < min_similarity:
        return {
            'eter_id': None,
            'eter_name_english': None,
            'eter_city': None,
            'eter_nuts3': None,
            'distance_km': round(best['distance_km'], 2),
            'name_similarity': round(best['name_similarity'], 1),
            'combined_score': None,
            'match_method': 'low_similarity'
        }
    
    return {
        'eter_id': best['BAS.ETERID'],
        'eter_name_english': best['BAS.INSTNAMEENGL'],
        'eter_city': best['GEO.CITY'],
        'eter_nuts3': best['GEO.NUTS3'],
        'distance_km': round(best['distance_km'], 2),
        'name_similarity': round(best['name_similarity'], 1),
        'combined_score': round(best['combined_score'], 1),
        'match_method': 'location_fuzzy'
    }

print("✓ Matching functions defined")

In [ ]:
# =============================================================================
# CELL 7: Run Full Institutional Matching
# =============================================================================

print("\n" + "=" * 80)
print(f"MATCHING {len(erasmus_institutions):,} INSTITUTIONS TO ETER")
print("=" * 80)
print(f"Parameters: max_distance={MAX_DISTANCE_KM}km, min_similarity={MIN_SIMILARITY}%")
print(f"Estimated time: 15-30 minutes\n")

all_results = []
match_count = 0
no_eter_country = 0
no_nearby = 0
low_similarity = 0

# Progress bar
for idx, inst in tqdm(erasmus_institutions.iterrows(), 
                      total=len(erasmus_institutions), 
                      desc="Matching", 
                      unit="inst"):
    
    match = find_best_eter_match(inst, df_eter_geo)
    
    # Track match statistics
    if match['eter_id']:
        match_count += 1
    elif match['match_method'] == 'no_eter_in_country':
        no_eter_country += 1
    elif match['match_method'] == 'no_nearby_eter':
        no_nearby += 1
    elif match['match_method'] == 'low_similarity':
        low_similarity += 1
    
    all_results.append({
        'erasmus_institution': inst['Institution'],
        'erasmus_city': inst['City'],
        'erasmus_country': inst['Country_Code'],
        'erasmus_lat': inst['Latitude'],
        'erasmus_lon': inst['Longitude'],
        'erasmus_nuts3': inst['NUTS3'],
        **match
    })

# Create results dataframe
results_linkage = pd.DataFrame(all_results)

print("\n" + "=" * 80)
print("MATCHING COMPLETE")
print("=" * 80)
print(f"Total institutions: {len(results_linkage):,}")
print(f"✓ Matched to ETER: {match_count:,} ({match_count/len(results_linkage)*100:.1f}%)")
print(f"  No ETER in country: {no_eter_country:,}")
print(f"  No nearby ETER (<{MAX_DISTANCE_KM}km): {no_nearby:,}")
print(f"  Low similarity (<{MIN_SIMILARITY}%): {low_similarity:,}")

In [ ]:
# =============================================================================
# CELL 8: Analyze Match Quality
# =============================================================================

print("\n" + "=" * 80)
print("MATCH QUALITY ANALYSIS")
print("=" * 80)

matched = results_linkage[results_linkage['eter_id'].notna()]

print(f"\n📊 MATCHED INSTITUTIONS ({len(matched):,}):")
print(f"  Average distance: {matched['distance_km'].mean():.2f} km")
print(f"  Median distance: {matched['distance_km'].median():.2f} km")
print(f"  Average name similarity: {matched['name_similarity'].mean():.1f}%")
print(f"  Median name similarity: {matched['name_similarity'].median():.1f}%")
print(f"  Average combined score: {matched['combined_score'].mean():.1f}")

# Distance distribution
print(f"\n📏 DISTANCE DISTRIBUTION:")
print(f"  Within 1 km: {(matched['distance_km'] < 1).sum():,} " +
      f"({(matched['distance_km'] < 1).sum()/len(matched)*100:.1f}%)")
print(f"  Within 5 km: {(matched['distance_km'] < 5).sum():,} " +
      f"({(matched['distance_km'] < 5).sum()/len(matched)*100:.1f}%)")
print(f"  Within 10 km: {(matched['distance_km'] < 10).sum():,} " +
      f"({(matched['distance_km'] < 10).sum()/len(matched)*100:.1f}%)")

# Similarity distribution
print(f"\n📝 NAME SIMILARITY DISTRIBUTION:")
print(f"  80-100%: {(matched['name_similarity'] >= 80).sum():,} " +
      f"({(matched['name_similarity'] >= 80).sum()/len(matched)*100:.1f}%)")
print(f"  70-79%: {((matched['name_similarity'] >= 70) & (matched['name_similarity'] < 80)).sum():,}")
print(f"  60-69%: {((matched['name_similarity'] >= 60) & (matched['name_similarity'] < 70)).sum():,}")

# Top countries by match rate
print(f"\n🌍 TOP 10 COUNTRIES BY MATCH RATE:")
country_stats = results_linkage.groupby('erasmus_country').agg({
    'eter_id': ['count', lambda x: x.notna().sum()]
})
country_stats.columns = ['total', 'matched']
country_stats['match_rate'] = (country_stats['matched'] / country_stats['total'] * 100).round(1)
country_stats = country_stats.sort_values('matched', ascending=False).head(10)
print(country_stats.to_string())

In [ ]:
# =============================================================================
# CELL 9: Save Linkage Table
# =============================================================================

print("\n" + "=" * 80)
print("SAVING LINKAGE TABLE")
print("=" * 80)

output_linkage = f"{OUTPUT_DIR}/erasmus_eter_linkage.csv"
results_linkage.to_csv(output_linkage, index=False)

print(f"✓ Linkage table saved: erasmus_eter_linkage.csv")
print(f"  Location: {output_linkage}")
print(f"  Records: {len(results_linkage):,}")
print(f"  Matched: {match_count:,}")

In [ ]:
# =============================================================================
# CELL 10: Merge Linkage Back to Main Dataset
# =============================================================================

print("\n" + "=" * 80)
print("MERGING ETER LINKAGE TO MAIN DATASET")
print("=" * 80)

# Reload main dataset
df_main = pd.read_csv(ERASMUS_CORRECTED_PATH)
print(f"Main dataset: {len(df_main):,} mobility records")

# Add full country format to linkage for matching
country_mapping = df_main[['Sending Country']].drop_duplicates()
country_mapping['Country_Code'] = country_mapping['Sending Country'].str.split(' - ').str[0]
country_map_dict = country_mapping.set_index('Country_Code')['Sending Country'].to_dict()

results_linkage['erasmus_country_full'] = results_linkage['erasmus_country'].map(country_map_dict)

# Merge for SENDING institutions
sending_linkage = results_linkage[[
    'erasmus_institution', 'erasmus_city', 'erasmus_country_full', 
    'eter_id', 'eter_name_english', 'eter_nuts3'
]].copy()
sending_linkage.columns = [
    'Sending Organization', 'Sending City', 'Sending Country',
    'Sending_ETER_ID', 'Sending_ETER_Name', 'Sending_ETER_NUTS3'
]

df_main = df_main.merge(
    sending_linkage, 
    on=['Sending Organization', 'Sending City', 'Sending Country'],
    how='left'
)

print(f"After sending merge: {len(df_main):,} records")

# Merge for RECEIVING institutions
receiving_linkage = results_linkage[[
    'erasmus_institution', 'erasmus_city', 'erasmus_country_full',
    'eter_id', 'eter_name_english', 'eter_nuts3'
]].copy()
receiving_linkage.columns = [
    'Receiving Organization', 'Receiving City', 'Receiving Country',
    'Receiving_ETER_ID', 'Receiving_ETER_Name', 'Receiving_ETER_NUTS3'
]

df_main = df_main.merge(
    receiving_linkage,
    on=['Receiving Organization', 'Receiving City', 'Receiving Country'],
    how='left'
)

print(f"After receiving merge: {len(df_main):,} records")

# Verify row count unchanged
if len(df_main) != len(pd.read_csv(ERASMUS_CORRECTED_PATH)):
    print("⚠️  WARNING: Row count changed during merge!")
else:
    print("✓ Row count unchanged")

In [ ]:
# =============================================================================
# CELL 11: Enrich with ETER Institutional Variables
# =============================================================================

print("\n" + "=" * 80)
print("ENRICHING WITH ETER INSTITUTIONAL DATA")
print("=" * 80)

# Use only latest year of ETER data to avoid duplicates
df_eter_latest = df_eter.sort_values('BAS.REFYEAR', ascending=False).drop_duplicates(
    'BAS.ETERID', keep='first'
)

print(f"ETER latest year: {len(df_eter_latest):,} institutions")
print(f"Reference years: {df_eter_latest['BAS.REFYEAR'].value_counts().to_dict()}")

# Select key ETER variables to merge
eter_vars_to_merge = [
    'BAS.REFYEAR',
    'PERS.ACAFTETOTAL',  # Academic staff FTE
    'REV.CORETOTAL.EURO',  # Core revenue
    'REV.THIRDPARTYTOTAL.EURO',  # Third-party revenue
    'PATPUBL.PUBLFULLTOT'  # Publications
]

# Check which variables are available
available_vars = [v for v in eter_vars_to_merge if v in df_eter_latest.columns]
print(f"Available ETER variables: {available_vars}")

# Prepare SENDING enrichment
eter_sending = df_eter_latest[['BAS.ETERID'] + available_vars].copy()
eter_sending.columns = ['Sending_ETER_ID'] + [f'Sending_ETER_{col}' for col in available_vars]

# Prepare RECEIVING enrichment
eter_receiving = df_eter_latest[['BAS.ETERID'] + available_vars].copy()
eter_receiving.columns = ['Receiving_ETER_ID'] + [f'Receiving_ETER_{col}' for col in available_vars]

# Merge
print("\nMerging ETER institutional variables...")
df_main = df_main.merge(eter_sending, on='Sending_ETER_ID', how='left')
df_main = df_main.merge(eter_receiving, on='Receiving_ETER_ID', how='left')

print(f"After ETER enrichment: {len(df_main):,} records")

if len(df_main) != len(pd.read_csv(ERASMUS_CORRECTED_PATH)):
    print("⚠️  WARNING: Row count changed!")
else:
    print("✓ Row count correct")

In [ ]:
# =============================================================================
# CELL 12: Save Final Dataset and Generate Summary
# =============================================================================

print("\n" + "=" * 80)
print("SAVING FINAL ENRICHED DATASET")
print("=" * 80)

output_file = f"{OUTPUT_DIR}/erasmus_with_eter.csv"
df_main.to_csv(output_file, index=False)

print(f"✓ Final dataset saved: erasmus_with_eter.csv")
print(f"  Location: {output_file}")
print(f"  Records: {len(df_main):,}")
print(f"  Columns: {len(df_main.columns)}")

# Final summary
print("\n" + "=" * 80)
print("ETER ENRICHMENT SUMMARY")
print("=" * 80)

print(f"\n📊 COVERAGE:")
sending_eter = df_main['Sending_ETER_ID'].notna().sum()
receiving_eter = df_main['Receiving_ETER_ID'].notna().sum()
both_eter = (df_main['Sending_ETER_ID'].notna() & df_main['Receiving_ETER_ID'].notna()).sum()

print(f"  Sending with ETER:   {sending_eter:,} / {len(df_main):,} " +
      f"({sending_eter/len(df_main)*100:.1f}%)")
print(f"  Receiving with ETER: {receiving_eter:,} / {len(df_main):,} " +
      f"({receiving_eter/len(df_main)*100:.1f}%)")
print(f"  Both with ETER:      {both_eter:,} / {len(df_main):,} " +
      f"({both_eter/len(df_main)*100:.1f}%)")

# Show sample
print(f"\n📋 SAMPLE WITH ETER DATA:")
sample = df_main[
    df_main['Sending_ETER_ID'].notna() & 
    df_main['Receiving_ETER_ID'].notna()
].head(3)

sample_cols = [
    'Sending Organization', 'Sending_ETER_Name',
    'Receiving Organization', 'Receiving_ETER_Name'
]
print(sample[sample_cols].to_string(index=False))

print("\n" + "=" * 80)
print("✅ ETER INSTITUTIONAL LINKAGE COMPLETE")
print("=" * 80)